# Retail Demand Forecasting - Student 3 (Google Colab version)
Spark runs in local mode in Colab, and files are saved under `/content` instead of HDFS. Run the cells from top to bottom.

## Step 1: Install PySpark and upload the CSV (run these two cells first)
When the upload box appears, choose `retail_demand.csv`.

In [1]:
!pip install -q pyspark

In [2]:
from google.colab import files
uploaded = files.upload()   # select retail_demand.csv
import os; print(os.listdir('/content'))

Saving retail_demand.csv to retail_demand.csv
['.config', 'retail_demand.csv', 'sample_data']


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, DoubleType)
from pyspark.sql import functions as F
from pyspark.sql.window import Window

### 0. Config

In [4]:
# 0. CONFIG (Google Colab: files are stored in /content, not HDFS)
RAW_PATH = "/content/retail_demand.csv"
PROCESSED_PATH = "/content/retail/processed/retail_clean"
RESULTS_BASE = "/content/retail/results"


### 1. Spark Session

In [5]:
# 1. SPARK SESSION
spark = (SparkSession.builder
         .appName("RetailDemandForecasting_Student3")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Master       :", spark.sparkContext.master)

Spark version: 4.0.4
Master       : local[*]


### 2. Read From Hdfs With An Explicit Schema

In [6]:
# 2. READ FROM HDFS WITH AN EXPLICIT SCHEMA
schema = StructType([
    StructField("Date", StringType(), True),
    StructField("Store ID", StringType(), True),
    StructField("Product ID", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Region", StringType(), True),
    StructField("Inventory Level", IntegerType(), True),
    StructField("Units Sold", IntegerType(), True),
    StructField("Units Ordered", IntegerType(), True),
    StructField("Price", DoubleType(), True),
    StructField("Discount", IntegerType(), True),
    StructField("Weather Condition", StringType(), True),
    StructField("Promotion", IntegerType(), True),
    StructField("Competitor Pricing", DoubleType(), True),
    StructField("Seasonality", StringType(), True),
    StructField("Epidemic", IntegerType(), True),
    StructField("Demand", IntegerType(), True),
])

raw = (spark.read
       .option("header", True)
       .schema(schema)
       .csv(RAW_PATH))

# snake_case column names (same naming as Student 2's Hive table)
df = raw
for c in raw.columns:
    df = df.withColumnRenamed(c, c.strip().lower().replace(" ", "_"))

df.printSchema()
print("Record count :", df.count())          # expected 76,000
print("Partitions   :", df.rdd.getNumPartitions())
df.show(5, truncate=False)

root
 |-- date: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- region: string (nullable = true)
 |-- inventory_level: integer (nullable = true)
 |-- units_sold: integer (nullable = true)
 |-- units_ordered: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- discount: integer (nullable = true)
 |-- weather_condition: string (nullable = true)
 |-- promotion: integer (nullable = true)
 |-- competitor_pricing: double (nullable = true)
 |-- seasonality: string (nullable = true)
 |-- epidemic: integer (nullable = true)
 |-- demand: integer (nullable = true)

Record count : 76000
Partitions   : 2
+----------+--------+----------+-----------+------+---------------+----------+-------------+-----+--------+-----------------+---------+------------------+-----------+--------+------+
|date      |store_id|product_id|category   |region|inventory_level|units_sold|units_ordered|price|

### 3. Data Quality Checks

In [7]:
# 3. DATA QUALITY CHECKS
print("--- Null count per column ---")
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c)
           for c in df.columns]).show()

print("--- Full-row duplicates / key duplicates ---")
total = df.count()
print("Distinct rows            :", df.distinct().count(), "of", total)
print("Distinct (date,store,prod):",
      df.select("date", "store_id", "product_id").distinct().count())

print("--- Invalid values ---")
print("Negative demand/units/price:",
      df.filter((F.col("demand") < 0) | (F.col("units_sold") < 0) |
                (F.col("units_ordered") < 0) | (F.col("price") <= 0)).count())
print("promotion not in (0,1)  :", df.filter(~F.col("promotion").isin(0, 1)).count())
print("epidemic  not in (0,1)  :", df.filter(~F.col("epidemic").isin(0, 1)).count())

print("--- Categorical value sets ---")
for c in ["category", "region", "weather_condition", "seasonality"]:
    print(c, [r[0] for r in df.select(c).distinct().orderBy(c).collect()])

df.select("demand", "units_sold", "units_ordered", "inventory_level",
          "price", "discount").describe().show()

--- Null count per column ---
+----+--------+----------+--------+------+---------------+----------+-------------+-----+--------+-----------------+---------+------------------+-----------+--------+------+
|date|store_id|product_id|category|region|inventory_level|units_sold|units_ordered|price|discount|weather_condition|promotion|competitor_pricing|seasonality|epidemic|demand|
+----+--------+----------+--------+------+---------------+----------+-------------+-----+--------+-----------------+---------+------------------+-----------+--------+------+
|   0|       0|         0|       0|     0|              0|         0|            0|    0|       0|                0|        0|                 0|          0|       0|     0|
+----+--------+----------+--------+------+---------------+----------+-------------+-----+--------+-----------------+---------+------------------+-----------+--------+------+

--- Full-row duplicates / key duplicates ---
Distinct rows            : 76000 of 76000
Distinct (da

### 4. Cleaning + Feature Engineering

In [8]:
# 4. CLEANING + FEATURE ENGINEERING
df = (df.withColumn("date", F.to_date("date", "yyyy-MM-dd"))
        .withColumn("year", F.year("date"))
        .withColumn("quarter", F.quarter("date"))
        .withColumn("month", F.month("date"))
        .withColumn("day_of_week", F.dayofweek("date"))
        .withColumn("week_of_year", F.weekofyear("date"))
        # unmet demand: positive = demand was higher than what was sold
        .withColumn("demand_gap", F.col("demand") - F.col("units_sold"))
        .withColumn("price_vs_competitor",
                    F.round(F.col("price") - F.col("competitor_pricing"), 2))
        .withColumn("effective_price",
                    F.round(F.col("price") * (1 - F.col("discount") / 100.0), 2)))

# Lag / rolling features per store-product series (window functions).
# Rolling windows end at the PREVIOUS row (-1) so the current demand
# is never used to predict itself (no target leakage).
w_series = Window.partitionBy("store_id", "product_id").orderBy("date")
w_roll7 = w_series.rowsBetween(-7, -1)
w_roll28 = w_series.rowsBetween(-28, -1)

df = (df.withColumn("demand_lag_1", F.lag("demand", 1).over(w_series))
        .withColumn("demand_lag_7", F.lag("demand", 7).over(w_series))
        .withColumn("demand_roll7_avg", F.round(F.avg("demand").over(w_roll7), 2))
        .withColumn("demand_roll28_avg", F.round(F.avg("demand").over(w_roll28), 2)))

df.cache()                      # reused by every analysis below
print("Rows after feature engineering:", df.count())
df.select("date", "store_id", "product_id", "demand", "demand_lag_1",
          "demand_roll7_avg", "demand_gap", "effective_price").show(8)

# Save the cleaned / engineered layer
(df.write.mode("overwrite").parquet(PROCESSED_PATH))
print("Written:", PROCESSED_PATH)


def save(result_df, name):
    """Write a small result table to HDFS as one CSV file."""
    (result_df.coalesce(1).write.mode("overwrite")
     .option("header", True).csv(f"{RESULTS_BASE}/{name}"))
    print("Written:", f"{RESULTS_BASE}/{name}")

Rows after feature engineering: 76000
+----------+--------+----------+------+------------+----------------+----------+---------------+
|      date|store_id|product_id|demand|demand_lag_1|demand_roll7_avg|demand_gap|effective_price|
+----------+--------+----------+------+------------+----------------+----------+---------------+
|2022-01-01|    S001|     P0017|    75|        NULL|            NULL|        16|          21.21|
|2022-01-02|    S001|     P0017|    42|          75|            75.0|         9|          15.98|
|2022-01-03|    S001|     P0017|    80|          42|            58.5|        21|          22.74|
|2022-01-04|    S001|     P0017|    62|          80|           65.67|         3|          16.41|
|2022-01-05|    S001|     P0017|    57|          62|           64.75|        21|          16.12|
|2022-01-06|    S001|     P0017|   132|          57|            63.2|        54|          21.82|
|2022-01-07|    S001|     P0017|   102|         132|           74.67|        23|         

### 5. Distributed Analytics (Dataframe Api)

In [9]:
# 5. DISTRIBUTED ANALYTICS (DataFrame API)
#    (Student 2 already covered category, region x season, weather x promo
#     in Hive, so these are different questions.)
# A1. Store performance
store_summary = (df.groupBy("store_id", "region")
                   .agg(F.count("*").alias("records"),
                        F.sum("demand").alias("total_demand"),
                        F.round(F.avg("demand"), 2).alias("avg_demand"),
                        F.round(F.avg("demand_gap"), 2).alias("avg_demand_gap"))
                   .orderBy(F.desc("total_demand")))
store_summary.show()
save(store_summary, "store_summary")

# A2. Quarterly demand trend
quarter_trend = (df.groupBy("year", "quarter")
                   .agg(F.sum("demand").alias("total_demand"),
                        F.round(F.avg("demand"), 2).alias("avg_demand"))
                   .orderBy("year", "quarter"))
quarter_trend.show()
save(quarter_trend, "quarter_trend")

# A3. Promotion lift by category
promo_lift = (df.groupBy("category")
                .pivot("promotion", [0, 1])
                .agg(F.round(F.avg("demand"), 2))
                .withColumnRenamed("0", "avg_demand_no_promo")
                .withColumnRenamed("1", "avg_demand_promo")
                .withColumn("promo_lift_pct",
                            F.round((F.col("avg_demand_promo") /
                                     F.col("avg_demand_no_promo") - 1) * 100, 1))
                .orderBy(F.desc("promo_lift_pct")))
promo_lift.show()
save(promo_lift, "promotion_lift")

# A4. Epidemic impact
epidemic_impact = (df.groupBy("epidemic")
                     .agg(F.count("*").alias("records"),
                          F.round(F.avg("demand"), 2).alias("avg_demand"))
                     .orderBy("epidemic"))
epidemic_impact.show()
save(epidemic_impact, "epidemic_impact")

# A5. Discount level vs demand
discount_effect = (df.groupBy("discount")
                     .agg(F.count("*").alias("records"),
                          F.round(F.avg("demand"), 2).alias("avg_demand"))
                     .orderBy("discount"))
discount_effect.show()
save(discount_effect, "discount_effect")

# A6. Unmet demand by category
unmet_demand = (df.groupBy("category")
                  .agg(F.round(F.avg("demand_gap"), 2).alias("avg_demand_gap"),
                       F.round(F.avg((F.col("demand_gap") > 0).cast("int")) * 100, 2)
                        .alias("pct_records_demand_exceeds_sales"))
                  .orderBy(F.desc("avg_demand_gap")))
unmet_demand.show()
save(unmet_demand, "unmet_demand")

# A7. Top 3 products per region (window ranking)
prod_region = (df.groupBy("region", "product_id")
                 .agg(F.sum("demand").alias("total_demand")))
w_rank = Window.partitionBy("region").orderBy(F.desc("total_demand"))
top_products = (prod_region.withColumn("rank", F.row_number().over(w_rank))
                           .filter(F.col("rank") <= 3)
                           .orderBy("region", "rank"))
top_products.show(12)
save(top_products, "top_products_per_region")

+--------+------+-------+------------+----------+--------------+
|store_id|region|records|total_demand|avg_demand|avg_demand_gap|
+--------+------+-------+------------+----------+--------------+
|    S002| South|  15200|     1625471|    106.94|         15.75|
|    S003|  East|  15200|     1618324|    106.47|         15.97|
|    S005| North|  15200|     1607085|    105.73|         16.07|
|    S001| North|  15200|     1547573|    101.81|         15.09|
|    S004|  West|  15200|     1529651|    100.63|         14.57|
+--------+------+-------+------------+----------+--------------+

Written: /content/retail/results/store_summary
+----+-------+------------+----------+
|year|quarter|total_demand|avg_demand|
+----+-------+------------+----------+
|2022|      1|     1014600|    112.73|
|2022|      2|      889913|     97.79|
|2022|      3|      954437|    103.74|
|2022|      4|     1003702|     109.1|
|2023|      1|      891357|     99.04|
|2023|      2|      907494|     99.72|
|2023|      3|  

### 6. Spark Sql (Shows Hive-Style Sql Running On Spark)

In [10]:
# 6. SPARK SQL (shows Hive-style SQL running on Spark)
df.createOrReplaceTempView("retail_clean")
spark.sql("""
    SELECT weather_condition,
           ROUND(AVG(demand), 2)     AS avg_demand,
           ROUND(AVG(demand_gap), 2) AS avg_demand_gap
    FROM retail_clean
    GROUP BY weather_condition
    ORDER BY avg_demand DESC
""").show()

+-----------------+----------+--------------+
|weather_condition|avg_demand|avg_demand_gap|
+-----------------+----------+--------------+
|            Sunny|    115.17|         18.14|
|           Cloudy|     105.4|         15.78|
|            Rainy|     95.11|         13.13|
|            Snowy|     94.05|          13.1|
+-----------------+----------+--------------+



### 7. Demand Forecasting With Spark Mllib

In [11]:
# 7. DEMAND FORECASTING WITH SPARK MLLIB
#    Only information known BEFORE the day is used:
#    price / discount / promotion / competitor price / weather / season /
#    epidemic / calendar / identifiers + past-demand features.
#    units_sold, units_ordered and inventory_level are excluded because they
#    are recorded on the same day as demand (target leakage).
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

cat_cols = ["store_id", "product_id", "category", "region",
            "weather_condition", "seasonality"]
num_cols = ["price", "discount", "promotion", "competitor_pricing", "epidemic",
            "month", "day_of_week", "week_of_year",
            "demand_lag_1", "demand_lag_7", "demand_roll7_avg", "demand_roll28_avg"]

model_df = df.dropna(subset=["demand_lag_1", "demand_lag_7",
                             "demand_roll7_avg", "demand_roll28_avg"])

# Time-based split (no random shuffling for time series)
CUTOFF = "2023-10-01"
train = model_df.filter(F.col("date") < CUTOFF)
test = model_df.filter(F.col("date") >= CUTOFF)
print("Train rows:", train.count(), "| Test rows:", test.count())

indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
            for c in cat_cols]
encoder = OneHotEncoder(inputCols=[f"{c}_idx" for c in cat_cols],
                        outputCols=[f"{c}_ohe" for c in cat_cols])
assembler = VectorAssembler(inputCols=[f"{c}_ohe" for c in cat_cols] + num_cols,
                            outputCol="features")

lr = LinearRegression(featuresCol="features", labelCol="demand")
rf = RandomForestRegressor(featuresCol="features", labelCol="demand",
                           numTrees=50, maxDepth=8, seed=42)

evaluators = {m: RegressionEvaluator(labelCol="demand", predictionCol="prediction",
                                     metricName=m) for m in ["rmse", "mae", "r2"]}

metrics = []

# Naive baseline: tomorrow's demand = yesterday's demand
base = test.withColumn("prediction", F.col("demand_lag_1").cast("double"))
metrics.append(("Baseline (lag-1)",
                *[evaluators[m].evaluate(base) for m in ["rmse", "mae", "r2"]]))

fitted = {}
for name, algo in [("Linear Regression", lr), ("Random Forest", rf)]:
    pipe = Pipeline(stages=indexers + [encoder, assembler, algo])
    model = pipe.fit(train)
    pred = model.transform(test)
    fitted[name] = (model, pred)
    metrics.append((name, *[evaluators[m].evaluate(pred) for m in ["rmse", "mae", "r2"]]))

metrics_df = spark.createDataFrame(
    [(n, round(a, 3), round(b, 3), round(c, 4)) for n, a, b, c in metrics],
    ["model", "rmse", "mae", "r2"])
metrics_df.show(truncate=False)
save(metrics_df, "model_metrics")

# Sample predictions + feature importance of the Random Forest
rf_model, rf_pred = fitted["Random Forest"]
rf_pred.select("date", "store_id", "product_id", "demand",
               F.round("prediction", 1).alias("predicted_demand")).show(10)
save(rf_pred.select("date", "store_id", "product_id", "demand",
                    F.round("prediction", 2).alias("predicted_demand")),
     "rf_predictions")

# map importances back to feature names using the assembler's metadata
attrs = rf_pred.schema["features"].metadata["ml_attr"]["attrs"]
names = [a["name"] for a in sorted((a for g in attrs.values() for a in g),
                                    key=lambda a: a["idx"])]
imp = rf_model.stages[-1].featureImportances.toArray()
imp_df = spark.createDataFrame(
    sorted(zip(names, [float(round(x, 4)) for x in imp]), key=lambda t: -t[1])[:15],
    ["feature", "importance"])
imp_df.show(truncate=False)
save(imp_df, "rf_feature_importance")

Train rows: 63100 | Test rows: 12200
+-----------------+------+------+------+
|model            |rmse  |mae   |r2    |
+-----------------+------+------+------+
|Baseline (lag-1) |51.465|39.671|-0.312|
|Linear Regression|32.79 |25.374|0.4674|
|Random Forest    |31.931|24.715|0.495 |
+-----------------+------+------+------+

Written: /content/retail/results/model_metrics
+----------+--------+----------+------+----------------+
|      date|store_id|product_id|demand|predicted_demand|
+----------+--------+----------+------+----------------+
|2023-10-01|    S001|     P0017|    57|            88.6|
|2023-10-02|    S001|     P0017|   115|           103.7|
|2023-10-03|    S001|     P0017|    85|            96.2|
|2023-10-04|    S001|     P0017|    77|            95.0|
|2023-10-05|    S001|     P0017|    77|            90.6|
|2023-10-06|    S001|     P0017|    72|            95.2|
|2023-10-07|    S001|     P0017|    24|            44.5|
|2023-10-08|    S001|     P0017|    41|            56.7|
|

### 8. Verify Outputs In Hdfs, Then Stop

In [12]:
# 8. VERIFY OUTPUTS IN HDFS, THEN STOP
#    (run from a terminal:  hdfs dfs -ls -R /data/retail/results )
print("Spark UI: http://localhost:4040  (take a screenshot while a job is running)")
df.unpersist()
spark.stop()

Spark UI: http://localhost:4040  (take a screenshot while a job is running)
